In [1]:
"""
cv_final.py: 5-fold cross-validation of the final model, to report a reliable
F1 (mean +- std) in the report.

The two parts come from the shared files, so everything stays consistent:
  * validation.py -> folds (stratified, same as the rest of the group),
                     threshold choice and metrics (cross_validate)
  * run.py        -> features (build_features) and model (train, predict_proba),
                     i.e. exactly the pipeline of the submission

In each fold, everything is learned on the training folds only: cleaning
rules, one-hot categories, standardization, weights and the threshold
(threshold="inner": chosen on an inner split, then the model is retrained
on all the training folds, as in run.py).

Usage: python cv_final.py   (about 5 minutes)
"""
import numpy as np

from run import load_data, build_features, train, predict_proba
from validation import cross_validate, print_cv_summary

K = 5
SEED = 0


def main():
    x_train, _, y01, _, names = load_data()

    def preprocess(x_tr, x_ev):
        """Features of run.py, learned on x_tr and applied to x_ev."""
        return build_features(x_tr, x_ev, names)

    def fit_predict(x_tr, y_tr, x_ev, seed):
        """Model of run.py: train on (x_tr, y_tr), return probabilities on x_ev."""
        w, mean, std = train(x_tr, y_tr.astype(float))
        return predict_proba(x_ev, w, mean, std)

    result = cross_validate(x_train, y01, fit_predict, n_splits=K, seed=SEED,
                            preprocess=preprocess, threshold="inner", verbose=True)
    print()
    print_cv_summary(result)
    print(f"Accuracy moyenne : {result['mean']['accuracy']:.4f} ; "
          f"écart-type : {result['std']['accuracy']:.4f}")


if __name__ == "__main__":
    main()

Pli 1/5 : F1=0.4205, seuil=0.20168
Pli 2/5 : F1=0.4312, seuil=0.197634
Pli 3/5 : F1=0.4259, seuil=0.213143
Pli 4/5 : F1=0.4283, seuil=0.175731
Pli 5/5 : F1=0.4281, seuil=0.210304

Pli   F1       Précision  Rappel    Accuracy  Seuil
  1   0.4205   0.3467     0.5344    0.8699    0.20168
  2   0.4312   0.3523     0.5555    0.8706    0.197634
  3   0.4259   0.3681     0.5053    0.8797    0.213143
  4   0.4283   0.3330     0.6003    0.8585    0.175731
  5   0.4281   0.3599     0.5282    0.8754    0.210304
F1 moyen : 0.4268 ; écart-type : 0.0036
F1 OOF agrégé : 0.4268
Accuracy moyenne : 0.8708 ; écart-type : 0.0071
